# Build OHIF bundles only (throwaway notebook)

Standalone, minimal notebook: just re-stages the small inputs needed and runs the
two new bundle-builder scripts against your **already-computed** results from the
main `monai_pathology_experts_colab.ipynb` run (reads the saved manifest on Drive —
no re-inference except one single forward pass for the brain-tumour demo case).

Delete this notebook after you've downloaded the two zips.

Runtime: **CPU is fine.** GPU (L4) only makes the brain-tumour single-case forward
pass faster -- it isn't required.

In [ ]:
%cd /content
!rm -rf doctor_assistant
!git clone -b classifier-evaluation-colab https://github.com/Hamza09Hamza/doctor_assistant.git
%cd doctor_assistant
%cd /content/doctor_assistant
!git pull origin classifier-evaluation-colab


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'monai==1.6.0', 'nibabel', 'einops',
                'pydicom', 'SimpleITK', 'itk',
                'idc-index', 'highdicom'], check=True)

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Running on CPU -- fine here (one single forward pass, no full training run).')

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

ROOT   = Path('/content/drive/MyDrive/doctor_assistant/monai_experts')
DATA   = ROOT / 'data'
OUTPUT = ROOT / 'results'
SCRATCH = Path('/content/monai_scratch')
SCRATCH.mkdir(parents=True, exist_ok=True)

manifest_path = OUTPUT / 'monai_experts_manifest.json'
assert manifest_path.exists(), (
    f'{manifest_path} not found -- the main notebook run must complete first '
    '(this notebook only builds viewer bundles from its already-saved results).'
)
print('Found manifest:', manifest_path)

## Lung nodule bundle

Re-stages the small single-case CT (fast, local scratch only -- not cached to Drive
by the main pipeline) using the same staging function `run_monai_pathology_experts.py`
uses, then rasterizes AI detections + ground truth from the existing manifest.

In [ ]:
import sys
sys.path.insert(0, '/content/doctor_assistant')
from types import SimpleNamespace
from scripts.run_monai_pathology_experts import _stage_lung_nodule_case

stage_args = SimpleNamespace(scratch_dir=SCRATCH)
ct_dir, seg_paths = _stage_lung_nodule_case(stage_args)
print('CT staged at:', ct_dir)

In [ ]:
import subprocess
_exit_code = subprocess.run(['python', 'scripts/build_lung_nodule_seg_bundle.py',
    '--ct-dir', '/content/monai_scratch/lung_nodule_case/ct',
    '--manifest', '/content/drive/MyDrive/doctor_assistant/monai_experts/results/monai_experts_manifest.json',
    '--output-dir', '/content/drive/MyDrive/doctor_assistant/monai_experts/results',
    '--top-n', '5', '--score-threshold', '0.3']).returncode
assert _exit_code == 0

## Brain tumour bundle

MSD data is Drive-cached already (`DATA`), so this just runs one single-case forward
pass and wraps it via the NIfTI->DICOM adapter.

In [1]:
%cd /content/doctor_assistant
!git pull origin classifier-evaluation-colab
!python scripts/run_lidc_lung_nodule_batch.py \
    --data-dir /content/drive/MyDrive/doctor_assistant/monai_experts/data \
    --scratch-dir /content/monai_scratch \
    --output-dir /content/drive/MyDrive/doctor_assistant/monai_experts/results \
    --max-cases 27 2>&1 | tee /content/drive/MyDrive/doctor_assistant/monai_experts/results/batch_log.txt


/content/doctor_assistant
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 4.12 KiB | 468.00 KiB/s, done.
From https://github.com/Hamza09Hamza/doctor_assistant
 * branch            classifier-evaluation-colab -> FETCH_HEAD
   fac36f5..46e345d  classifier-evaluation-colab -> origin/classifier-evaluation-colab
Updating fac36f5..46e345d
Fast-forward
 scripts/run_lidc_lung_nodule_batch.py    | 66 +++++++++++++++++++++++++++-----
 tests/test_run_lidc_lung_nodule_batch.py | 40 +++++++++++++++++++
 2 files changed, 97 insertions(+), 9 deletions(-)
[09:22:18] LUNA16 training/eval corpus: 888 unique series
[09:22:18] Discovered 27 clean candidate case(s).
[09:22:18] Running batch on 27 case(s).
[09:22:18] Staging 27 case(s) sequentially (1 worker) ...
[09:22:18]   LIDC-IDRI-0079: CT done (113 files)
[09:22:18]   L

## Done

Download `lung_nodule_ohif_bundle.zip` and the brain-tumour bundle zip from
`Drive/doctor_assistant/monai_experts/results/`, then locally:

```
python scripts/publish_dicom_seg_to_orthanc.py --bundle <lung_nodule_ohif_bundle.zip>
python scripts/publish_dicom_seg_to_orthanc.py --bundle <brain_tumor bundle zip>
```

Delete this notebook when done.